# Text Emotion Transfer Learning + MLflow (Kaggle-ready)

- Modèle pré-entraîné : `bhadresh-savani/distilbert-base-uncased-emotion` (rapide et bonne précision)

- Jeu de données : vos CSV `data/raw/train_text.csv` et `data/raw/test_text.csv`

- Tracking & registry : MLflow local (`/kaggle/working/mlruns` ou `./mlruns` en local)

- Étapes : install → préparation données → fine-tuning → éval → tracking → (optionnel) registry


In [ ]:
# 1) Install requirements (Kaggle/colab/local)
!pip -q install --upgrade transformers datasets accelerate evaluate mlflow scikit-learn matplotlib


In [ ]:
# 2) Imports & paths
import os, sys, json, random
import numpy as np
import pandas as pd
import mlflow
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt

# Add repo to path (Kaggle: /kaggle/working/<repo>)
REPO_ROOT = os.path.abspath('..')  # adjust if needed
sys.path.append(REPO_ROOT)

from mlops.mlflow import get_mlflow_tracker, TextModelTracker
from mlops.registry import get_model_registry, TextModelRegistry

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Data paths (local or Kaggle)
TRAIN_PATH = os.path.join(REPO_ROOT, 'data', 'raw', 'train_text.csv')
TEST_PATH  = os.path.join(REPO_ROOT, 'data', 'raw', 'test_text.csv')

# MLflow local store (Kaggle: /kaggle/working/mlruns)
TRACKING_URI = os.path.join(REPO_ROOT, 'mlruns')
os.makedirs(TRACKING_URI, exist_ok=True)


In [ ]:
# 3) Load data & prepare labels
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

# Build label list from train set
labels = sorted(train_df['emotion'].unique())
label2id = {lbl: i for i, lbl in enumerate(labels)}
id2label = {i: lbl for lbl, i in label2id.items()}
num_labels = len(labels)

print('Labels:', labels)

# Map labels to ids
train_df = train_df.assign(label=train_df['emotion'].map(label2id))
test_df  = test_df.assign(label=test_df['emotion'].map(label2id))

# Hugging Face datasets
datasets = DatasetDict({
    'train': Dataset.from_pandas(train_df[['transcription', 'label']]),
    'test':  Dataset.from_pandas(test_df[['transcription', 'label']])
})


In [ ]:
# 4) Tokenization & datasets
MODEL_NAME = 'bhadresh-savani/distilbert-base-uncased-emotion'  # rapide et précis

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(batch['transcription'], truncation=True)

encoded = datasets.map(tokenize_batch, batched=True)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

encoded = encoded.remove_columns(['transcription'])
encoded = encoded.rename_column('label', 'labels')
encoded.set_format('torch')


In [ ]:
# 5) Model & training setup
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

# Training hyperparams
EPOCHS = 3
LR = 2e-5
BATCH = 16

training_args = TrainingArguments(
    output_dir='outputs/text_model',
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=50,
    seed=SEED,
    report_to=['none'],  # we will log manually to MLflow
)

# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    return {'accuracy': acc, 'f1': f1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded['train'],
    eval_dataset=encoded['test'],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)


In [ ]:
# (Optional) Geler le backbone et n'entraîner que la tête de classification
# Pour DistilBERT : on garde trainables uniquement pre_classifier et classifier
for name, param in model.named_parameters():
    if name.startswith("pre_classifier") or name.startswith("classifier"):
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable = [n for n,p in model.named_parameters() if p.requires_grad]
frozen    = [n for n,p in model.named_parameters() if not p.requires_grad]
print(f"Trainable params: {len(trainable)}, frozen: {len(frozen)}")


In [ ]:
# 6) Fine-tuning
train_result = trainer.train()
metrics_train = train_result.metrics
print('Train metrics:', metrics_train)

eval_metrics = trainer.evaluate()
print('Eval metrics:', eval_metrics)


In [ ]:
# 7) Evaluation details: confusion matrix
preds_output = trainer.predict(encoded['test'])
preds = np.argmax(preds_output.predictions, axis=1)
true = preds_output.label_ids

cm = confusion_matrix(true, preds)
fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(num_labels)); ax.set_xticklabels(labels, rotation=45)
ax.set_yticks(range(num_labels)); ax.set_yticklabels(labels)
for i in range(num_labels):
    for j in range(num_labels):
        ax.text(j, i, cm[i, j], ha='center', va='center', color='black')
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('Confusion Matrix')
fig.colorbar(im)
os.makedirs('artifacts', exist_ok=True)
cm_path = 'artifacts/confusion_matrix.png'
plt.tight_layout(); plt.savefig(cm_path, dpi=120)
plt.close(fig)


In [ ]:
# 8) MLflow tracking + registry (local store)
tracker = get_mlflow_tracker(tracking_uri=TRACKING_URI, experiment_name='text_emotion')
text_tracker = TextModelTracker(tracker)
registry = get_model_registry(registry_uri=TRACKING_URI)
text_registry = TextModelRegistry(registry)

# Params & metrics to log
params = {
    'model_name': MODEL_NAME,
    'epochs': EPOCHS,
    'learning_rate': LR,
    'batch_size': BATCH,
    'num_labels': num_labels,
}
metrics = {
    'train_loss': float(metrics_train.get('train_loss', 0.0)),
    'eval_loss': float(eval_metrics.get('eval_loss', 0.0)),
    'eval_accuracy': float(eval_metrics.get('eval_accuracy', 0.0)),
    'eval_f1': float(eval_metrics.get('eval_f1', 0.0)),
}

# Start run
tracker.start_run(run_name=f"text_finetune_{MODEL_NAME.split('/')[-1]}", tags={'modality': 'text'})

# Log metadata & metrics
tracker.log_model_metadata(
    model_name='text_emotion_model',
    modality='text',
    architecture=MODEL_NAME,
    params=params,
)
tracker.log_metrics(metrics)

# Save model & log with MLflow transformers flavor
save_dir = 'artifacts/text_model'
trainer.save_model(save_dir)
mlflow.transformers.log_model(
    transformers_model={'model': model, 'tokenizer': tokenizer},
    artifact_path='model',
    task='text-classification'
)

# Log confusion matrix
tracker.log_artifact(cm_path, artifact_path='evaluation')

# Register model (optional)
run_id = mlflow.active_run().info.run_id
model_uri = f"runs:/{run_id}/model"
reg_uri = text_registry.register_version(model_uri, version_description='finetuned on custom dataset')
print('Registered model URI:', reg_uri)

# End run
tracker.end_run()


In [ ]:
# 9) Sauvegarder le modèle PyTorch pour réutilisation ultérieure
import torch

# Créer le dossier de sauvegarde
MODEL_SAVE_DIR = 'saved_models/text_emotion_pytorch'
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

# Sauvegarder le modèle PyTorch complet
model_path = os.path.join(MODEL_SAVE_DIR, 'model.pt')
torch.save(model.state_dict(), model_path)

# Sauvegarder aussi la config et le tokenizer
model.config.save_pretrained(MODEL_SAVE_DIR)
tokenizer.save_pretrained(MODEL_SAVE_DIR)

# Sauvegarder les labels mapping
import json
labels_info = {
    'labels': labels,
    'label2id': label2id,
    'id2label': id2label,
    'num_labels': num_labels
}
with open(os.path.join(MODEL_SAVE_DIR, 'labels.json'), 'w') as f:
    json.dump(labels_info, f, indent=2)

print(f"✓ Modèle PyTorch sauvegardé dans: {MODEL_SAVE_DIR}")
print(f"  - model.pt (poids du modèle)")
print(f"  - config.json (configuration)")
print(f"  - tokenizer files")
print(f"  - labels.json (mapping des labels)")


In [ ]:
# 10) Recharger le modèle PyTorch sauvegardé (pour inférence)
# Cette cellule montre comment recharger le modèle plus tard

from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification
import json

# Chemin du modèle sauvegardé
LOAD_MODEL_DIR = 'saved_models/text_emotion_pytorch'

# Charger les labels
with open(os.path.join(LOAD_MODEL_DIR, 'labels.json'), 'r') as f:
    labels_data = json.load(f)
    
loaded_labels = labels_data['labels']
loaded_label2id = {k: int(v) for k, v in labels_data['label2id'].items()}
loaded_id2label = {int(k): v for k, v in labels_data['id2label'].items()}

# Charger le modèle
loaded_config = AutoConfig.from_pretrained(LOAD_MODEL_DIR)
loaded_tokenizer = AutoTokenizer.from_pretrained(LOAD_MODEL_DIR)
loaded_model = AutoModelForSequenceClassification.from_pretrained(
    LOAD_MODEL_DIR,
    config=loaded_config,
    state_dict=torch.load(os.path.join(LOAD_MODEL_DIR, 'model.pt'))
)

# Test d'inférence
test_text = "I'm so happy today!"
inputs = loaded_tokenizer(test_text, return_tensors="pt", truncation=True, padding=True)

loaded_model.eval()
with torch.no_grad():
    outputs = loaded_model(**inputs)
    logits = outputs.logits
    pred_id = torch.argmax(logits, dim=1).item()
    probs = torch.softmax(logits, dim=1)[0]

predicted_emotion = loaded_id2label[pred_id]
confidence = probs[pred_id].item()

print(f"✓ Modèle rechargé depuis: {LOAD_MODEL_DIR}")
print(f"\nTest d'inférence:")
print(f"Texte: '{test_text}'")
print(f"Émotion prédite: {predicted_emotion} (confiance: {confidence:.4f})")
print(f"\nToutes les probabilités:")
for i, label in enumerate(loaded_labels):
    print(f"  {label}: {probs[i].item():.4f}")
